In [0]:
#

In [0]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

In [0]:
dbutils.widgets.text("evh_name", "artemzharkov10_evh")
dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("schema", "artemzharkov10_bronze")

EVH_NAME = dbutils.widgets.get("evh_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

In [0]:

EH_CONN_STR = dbutils.secrets.get(scope="default2", key="artem-evh-connector")
BOOTSTRAP = "evhpl24databricks.servicebus.windows.net:9093"
JAAS = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EH_CONN_STR}";'

checkpoint_path = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/checkpoints/bitcoin_stream_checkpoint"
landing_path = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/streaming_bitcoin" 



In [0]:
# creating a query with JAAS:(like a passport) to Azure Event Hab
raw_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", EVH_NAME) 
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", JAAS) # passport to enter in evh
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 100) # batch sizw
    .load()
)

In [0]:

# CAST - type converter, before 010010 after {asset:BTC};   
landing_df = raw_df.selectExpr("CAST(value AS STRING) as json_payload") # transform batch to understandble text

query = (
    landing_df.writeStream
    .format("text") # already JSON therefore format is text
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True) # (Cost Awareness!) end script if no new data in evh
    .start(landing_path)
)
